In [1]:
import xarray as xr
import pandas as pd
import numpy as np
from pandas.tseries.offsets import DateOffset

/home/jupyter-vincent2/.local/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/jupyter-vincent2/.local/lib/python3.10/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


In [2]:
### CMA
ds = xr.open_dataset("/home/jupyter-vincent2/vincent/process_profiles/data/All/all_CORA_2024.nc")
ds = ds.rename({"LATITUDE":"latitude", "LONGITUDE":"longitude"})
df = ds.mld.to_dataframe().reset_index() 

def make_ds_cut(df, nb_bins=39, freq="M"):
    # detect coordinate column names
    lon_col = "longitude"
    lat_col = "latitude"
    time_col = "time"

    bins_dt = pd.date_range(
        start=df[time_col].min() + DateOffset(months=-1),
        end=df[time_col].max() + DateOffset(months=+1),
        freq=freq
    )

    cut_lat_label = pd.cut(df[lat_col], nb_bins)
    cut_lon_label = pd.cut(df[lon_col], nb_bins)
    cut_time_label = pd.cut(df[time_col], bins=bins_dt)

    df_cut_label = df.drop([lat_col, lon_col, time_col], axis=1)
    df_cut_label = df_cut_label.groupby([cut_time_label, cut_lon_label, cut_lat_label]).mean()

    lat_mid = pd.IntervalIndex(df_cut_label.index.get_level_values(2)).mid.unique()
    lon_mid = pd.IntervalIndex(df_cut_label.index.get_level_values(1)).mid.unique()
    time_mid = pd.IntervalIndex(df_cut_label.index.get_level_values(0)).mid.unique()

    df_cut_label.index = df_cut_label.index.set_levels(time_mid.values, level=0)
    df_cut_label.index = df_cut_label.index.set_levels(lon_mid, level=1)
    df_cut_label.index = df_cut_label.index.set_levels(lat_mid.values, level=2)

    df_cut = df_cut_label.copy()
    df_cut.replace(0, np.nan, inplace=True)

    ds_cut = df_cut.to_xarray()
    ds_cut["latitude"] = sorted(lat_mid)
    ds_cut["longitude"] = sorted(lon_mid)
    ds_cut["time"] = time_mid

    return ds_cut, df_cut

ds, df = make_ds_cut(df, nb_bins=39)

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
/tmp/ipykernel_556327/2965905509.py:12: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  bins_dt = pd.date_range(
/tmp/ipykernel_556327/2965905509.py:23: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_cut_label = df_cut_label.groupby([cut_time_label, cut_lon_label, cut_lat_label]).mean()


In [3]:
ds = ds.where(ds.time.dt.year < 2024,drop=True)
df = ds.to_dataframe().reset_index()
### We modify the dataset for the R analysis 
df['year'] = df.time.dt.year
df["mth"] = df.time.dt.month
df = df.drop(columns="time")
df = df.rename(columns={"latitude" : "lat","longitude" : "long"})
df = df.fillna("NA")
df = df[["long","lat","mth","year","mld"]]
df = df.sort_values(["year","mth","long","lat"])
df = df.reset_index(drop=True)

In [4]:
### Number of bins for each month = nb_lat_one_month * nb_long_one_month 
nb_lat_one_month=len(np.unique(df.lat))
nb_long_one_month=len(np.unique(df.long))
nb_bins_one_month = nb_lat_one_month * nb_long_one_month
switch = nb_long_one_month*nb_lat_one_month
switch
df_append = df[:switch].copy()
new_df = df.copy()
new_df

,long,lat,mth,year,mld
0,60.2465,-59.7515,1,2007,NA
1,60.2465,-59.2290,1,2007,NA
2,60.2465,-58.7165,1,2007,NA
3,60.2465,-58.2035,1,2007,NA
4,60.2465,-57.6905,1,2007,NA
...,...,...,...,...,...
293548,79.7435,-42.3080,1,2023,NA
293549,79.7435,-41.7955,1,2023,NA
293550,79.7435,-41.2825,1,2023,NA
293551,79.7435,-40.7695,1,2023,NA


In [5]:
new_df.to_csv("/home/jupyter-vincent2/vincent/process_profiles/data/R_analysis_2026/raw/CORA.txt",sep=" ",index=False)
ds.to_netcdf("/home/jupyter-vincent2/vincent/process_profiles/data/processed_2026/CORA_gridded.nc")